In [1]:
import os
import shutil
import random

# Paths
input_dir = "PlantVillage"  
output_dir = "dataset"
train_dir = os.path.join(output_dir, "train")
test_dir = os.path.join(output_dir, "test")


os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

split_ratio = 0.8 

for category in os.listdir(input_dir):
    category_path = os.path.join(input_dir, category)
    if not os.path.isdir(category_path):
        continue

    images = os.listdir(category_path)
    random.shuffle(images)

    split_point = int(len(images) * split_ratio)
    train_images = images[:split_point]
    test_images = images[split_point:]

    os.makedirs(os.path.join(train_dir, category), exist_ok=True)
    os.makedirs(os.path.join(test_dir, category), exist_ok=True)

    for img in train_images:
        shutil.copy(os.path.join(category_path, img), os.path.join(train_dir, category, img))

    for img in test_images:
        shutil.copy(os.path.join(category_path, img), os.path.join(test_dir, category, img))

print("✅ Dataset split into train/test folders!")


✅ Dataset split into train/test folders!


In [2]:
import tensorflow as tf

print("✅ TensorFlow version:", tf.__version__)
print("Is GPU available?:", tf.config.list_physical_devices('GPU'))


✅ TensorFlow version: 2.20.0
Is GPU available?: []


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

test_data = test_datagen.flow_from_directory(
    "dataset/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)


Found 20638 images belonging to 15 classes.
Found 18903 images belonging to 15 classes.


In [4]:
train_data = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb"   # 👈 force 3 channels
)

test_data = test_datagen.flow_from_directory(
    "dataset/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb"   # 👈 force 3 channels
)


Found 20638 images belonging to 15 classes.
Found 18903 images belonging to 15 classes.


In [5]:
from tensorflow.keras.applications import EfficientNetB0


In [6]:
base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224,224,3))


In [7]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False


In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb"   
)

test_data = test_datagen.flow_from_directory(
    "dataset/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb"
)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze pre-trained layers

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(train_data.num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(train_data, validation_data=test_data, epochs=10)


model.save("plant_leaf_disease.h5")
print("✅ Model trained and saved as plant_leaf_disease.h5")


Found 20638 images belonging to 15 classes.
Found 18903 images belonging to 15 classes.


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 893s 1s/step - accuracy: 0.7497 - loss: 0.7611 - val_accuracy: 0.8694 - val_loss: 0.4109
Epoch 2/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 386s 599ms/step - accuracy: 0.8426 - loss: 0.4641 - val_accuracy: 0.8848 - val_loss: 0.3386
Epoch 3/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 370s 574ms/step - accuracy: 0.8665 - loss: 0.3965 - val_accuracy: 0.8965 - val_loss: 0.2953
Epoch 4/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 364s 565ms/step - accuracy: 0.8727 - loss: 0.3693 - val_accuracy: 0.8972 - val_loss: 0.2919
Epoch 5/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 362s 561ms/step - accuracy: 0.8820 - loss: 0.3483 - val_accuracy: 0.9167 - val_loss: 0.2436
Epoch 6/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 368s 571ms/step - accuracy: 0.8847 - loss: 0.3355 - val_accuracy: 0.8882 - val_loss: 0.3186
Epoch 7/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 372s 577ms/step - accuracy: 0.8945 - loss: 0.3108 - val_accuracy: 0.9194 - val_loss: 0.2272
Epoch 8/10
645/645 ━━━━━━━━━━━━━━━━━━━━ 344s 533ms/step - accuracy: 0.8976 - lo

✅ Model trained and saved as plant_leaf_disease.h5


In [9]:
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model("plant_leaf_disease.h5")

class_labels = list(model.class_names) if hasattr(model, "class_names") else None

if class_labels is None:

    class_labels = ["Apple___healthy", "Apple___rust", "Corn___blight", "Corn___healthy"]


st.title("🌿 Plant Leaf Disease Detection")
st.write("Upload a plant leaf image and the model will predict the disease.")

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:

    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)

    img = image.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

  
    predictions = model.predict(img_array)
    class_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0])

    
    st.subheader(f"✅ Predicted Disease: **{class_labels[class_index]}**")
    st.write(f"Confidence: {confidence*100:.2f}%")


2025-09-20 18:37:12.292 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:13.214 
  command:

    streamlit run C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-09-20 18:37:13.214 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:13.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:13.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:13.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:13.217 Thread 'MainThread': missing ScriptRunContext! This 

In [25]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model("plant_leaf_disease.h5")


class_labels = list(train_data.class_indices.keys())  # e.g., ["Apple___healthy", "Apple___rust", ...]

def predict_leaf(img_path):
  
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0 
    img_array = np.expand_dims(img_array, axis=0)  

 
    predictions = model.predict(img_array)
    class_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0])

    print(f"Predicted Disease: {class_labels[class_index]} ({confidence*100:.2f}% confidence)")
    return class_labels[class_index], confidence

predict_leaf(r"C:\Users\Lenovo\Downloads\archive (1)\PlantVillage\PlantVillage\Potato___Late_blight\0acdc2b2-0dde-4073-8542-6fca275ab974___RS_LB 4857.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 979ms/step
Predicted Disease: Potato___Late_blight (99.70% confidence)


('Potato___Late_blight', np.float32(0.99696904))

In [26]:
predict_leaf(r"C:\Users\Lenovo\Downloads\archive (1)\PlantVillage\PlantVillage\Tomato_Early_blight\0a2726e0-3358-4a46-b6dc-563a5a9f2bdf___RS_Erly.B 7860.JPG")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Predicted Disease: Tomato_Early_blight (52.19% confidence)


('Tomato_Early_blight', np.float32(0.5218684))

In [12]:
st.camera_input("Take a picture")


2025-09-20 18:37:14.708 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:14.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:14.711 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:14.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:14.713 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:14.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [13]:
model.save("plant_disease_model.h5")


In [24]:
model = tf.keras.models.load_model("plant_disease_model.h5")


In [23]:
model = tf.keras.models.load_model(r"C:\Users\Lenovo\Downloads\archive (1)\PlantVillage\plant_leaf_disease.h5")


In [18]:
import os
print(os.listdir())


['dataset', 'imagedetection.ipynb', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'PlantVillage', 'plant_disease_model.h5', 'plant_leaf_disease.h5', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_mosaic_virus', 'Tomato__Tomato_YellowLeaf__Curl_Virus']


In [17]:
import streamlit as st
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

model = tf.keras.models.load_model("plant_disease_model.h5")

class_labels = ["Potato___Late_blight", "Potato___Early_blight", "Potato___Healthy"]

def predict_leaf(img):
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction) * 100
    return class_labels[class_index], confidence

st.title("🌿 Plant Disease Detection")

option = st.radio("Choose input method:", ["Upload Image", "Use Camera"])

if option == "Upload Image":
    uploaded_file = st.file_uploader("Upload a leaf image", type=["jpg", "png", "jpeg"])
    if uploaded_file:
        img = image.load_img(uploaded_file, target_size=(224,224))
        st.image(img, caption="Uploaded Leaf", use_column_width=True)
        label, confidence = predict_leaf(img)
        st.success(f"Prediction: {label} ({confidence:.2f}%)")

elif option == "Use Camera":
    camera_photo = st.camera_input("Take a picture of the leaf")
    if camera_photo:
        img = image.load_img(camera_photo, target_size=(224,224))
        st.image(img, caption="Captured Leaf", use_column_width=True)
        label, confidence = predict_leaf(img)
        st.success(f"Prediction: {label} ({confidence:.2f}%)")


2025-09-20 18:37:16.392 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.392 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.396 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.397 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-20 18:37:16.397 Session state does not function when running a script without `streamlit run`
2025-09-20 18:37

In [28]:
predict_leaf(r"C:\Users\Lenovo\Downloads\archive (1)\PlantVillage\Potato___Early_blight\0a8a68ee-f587-4dea-beec-79d02e7d3fa4___RS_Early.B 8461.JPG")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Predicted Disease: Potato___Early_blight (99.98% confidence)


('Potato___Early_blight', np.float32(0.9998196))